In [ ]:
from dotenv import load_dotenv
import os
load_dotenv('../env')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
LANGSMITH_API_KEY=os.getenv('LANGSMITH_API_KEY')

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGCHAIN_PROJECT'] = 'LANG-1'

In [14]:
import utils

In [54]:
from pathlib import Path
import time
import sqlite3

DB_PATH = Path("data/titanic.db")

def db_ping(_ : str) -> str:
    
    """
    SQLite DB에 '연결만' 확인하는 경량 핑.
    - db_path: Path 객체 (존재해야 성공)
    - 반환: 'OK | ...' 또는 'FAIL | ...'
    """
    timeout = 1.5
    t0 = time.perf_counter()
    try:
        # 읽기 전용으로 열어 없으면 에러(= 허위 성공 방지)
        uri = f"file:{DB_PATH.as_posix()}?mode=ro"
        with sqlite3.connect(uri, uri=True, timeout=timeout) as conn:
            # 초경량 쿼리(옵션이지만 연결 확인에 유용)
            conn.execute("SELECT 1").fetchone()
        dt_ms = (time.perf_counter() - t0) * 1000
        return f"OK | DB_PATH={DB_PATH} | connect OK in {dt_ms:.1f} ms"
    except sqlite3.Error as e:
        return f"FAIL | DB_PATH={DB_PATH} | sqlite error: {e.__class__.__name__}: {e}"
    except Exception as e:
        return f"FAIL | DB_PATH={DB_PATH} | error: {e.__class__.__name__}: {e}"

def db_tables(_: str = "") -> str:
    try:
        with utils._conn(DB_PATH) as con:
            cur = con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
            rows = cur.fetchall()
            headers = ["table_name"]
            return utils._markdown_table(headers, rows) if rows else "_(no tables)_"
    except Exception as e:
        return f"ERROR: {e} | DB_PATH={DB_PATH}"

def db_schema(table: str = "passengers") -> str:
    t = utils._clean_table_name(table)
    try:
        with utils._conn(DB_PATH) as con:
            cur = con.execute(f"PRAGMA table_info({t});")
            rows = cur.fetchall()
            if not rows:
                return f"Table '{table}' not found in DB: {DB_PATH}"
            # PRAGMA table_info: (cid, name, type, notnull, dflt_value, pk)
            headers = ["ord", "column", "type", "notnull", "dflt_value", "pk"]
            return utils._markdown_table(headers, rows)
    except Exception as e:
        return f"ERROR: {e} | DB_PATH={DB_PATH}"

def db_query(sql: str) -> str:
    if not utils._is_select(sql):
        return "ERROR: Only SELECT/CTE allowed."
    q = utils._ensure_limit(sql, 500)
    try:
        with utils._conn(DB_PATH) as con:
            cur = con.execute(q)
            rows = cur.fetchall()
            headers = [d[0] for d in (cur.description or [])]
            return utils._markdown_table(headers, rows)
    except Exception as e:
        return f"ERROR: {e} | DB_PATH={DB_PATH}"

In [55]:
from pathlib import Path
print(db_ping(DB_PATH))

OK | DB_PATH=data/titanic.db | connect OK in 6.2 ms


In [57]:
from pathlib import Path
META_FN = os.getenv("META_FILE", "metadata.txt")

meta_text = Path(META_FN).read_text(encoding="utf-8").strip() if Path(META_FN).exists() else ""
SYSTEM = f"""당신은 SQLite 데이터베이스를 조회/요약하는 어시스턴트입니다.
오직 데이터베이스를 기반으로만 답변하고, 데이터베이스 외의 정보는 절대 제시하지 말아야 합니다.
중요 제약:
- 현재 데이터베이스는 복잡한 집계 함수와 GROUP BY 사용이 제한될 수 있습니다.
- pandas 및 DataFrame 사용 금지. 결과 표는 도구가 반환한 Markdown을 그대로 사용합니다.
- 제공 도구: list_tables, describe_table, sql_execute (SELECT는 자동 LIMIT 500 적용).
- 기본은 읽기 전용입니다. INSERT/UPDATE/DELETE/DDL은 사용할 수 없습니다.

요청 처리 원칙:
1) 데이터베이스 외의 정보는 절대 제시하지 말아야 합니다.
2) 데이터베이스 연결은 DBPing을 통해 확인하세요.
3) 가능한 한 **단순 SELECT**(필드 선택, WHERE, ORDER BY, LIMIT)로 분해해 실행하세요.
4) 사용자가 평균/중앙값/비율/랭킹 등 **복잡 집계**를 요구하면,
   - (가) 필요한 최소 컬럼만 SELECT하여 제한된 샘플(기본 500행) 결과를 가져온 뒤,
   - (나) 그 범위 안에서 **텍스트로 계산/추정**하거나, 정확치 않음(표본 기준)임을 분명히 밝히세요.
   - (다) 전체 정확도가 필요하면 “전체 스캔/청크 조회가 필요함”을 안내하고, 허용 시 청크 크기 및 최대 행 수를 제안하세요.
5) 쿼리는 항상 **LIMIT**을 유지합니다. LIMIT 확장이나 풀스캔이 필요하면 먼저 사용자 동의를 받습니다.
6) 결과 표현:
   - 먼저 “요약”을 한 줄로 제시하고, 이어서 **실행한 SQL**을 코드 블록으로 보여주며, 마지막에 표(도구 반환 Markdown)를 그대로 붙입니다.
   - 결과 행이 0이면 간단한 원인 가설(조건 과도, 컬럼명 오타 등)과 다음 시도 옵션을 제시하세요.
7) 스키마가 불명확하면 `list_tables` → `describe_table` 순으로 빠르게 확인 후 쿼리를 만드세요.
8) 비파괴성: 쓰기 작업은 “실행 전 확인”을 받고, 실행 후 `rows_affected`/`lastrowid`만 보고합니다. 롤백/트랜잭션 옵션은 사용자 요청 시에만 언급합니다.
9) 응답은 한국어로 간결하게. 내부 추론 과정은 노출하지 말고, 최종 결론만 보여주고 SQL를 마지막에 제시하세요.

[DATA DICTIONARY]
{meta_text}
"""

from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.1,
)

# import packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage
from langchain.agents import Tool, AgentType, initialize_agent
from langchain.memory import ConversationBufferMemory

tools = [
    Tool(name="DBPing",   func=db_ping,   description="연결 확인 및 DB 경로 확인."),
    Tool(name="DBTables", func=db_tables, description="사용 가능한 테이블 목록을 보여준다."),
    Tool(name="DBSchema", func=db_schema, description="테이블 컬럼/타입 확인(기본 'passengers')."),
    Tool(name="DBQuery",  func=db_query,  description="SQLite DB에서 SELECT/CTE 실행."),
]

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=False,
    agent_kwargs={"system_message": SystemMessage(content=SYSTEM)},
)

In [58]:
out = agent.invoke({"input": '남자의 수는?'})
print("\n답변 >\n", out["output"])


답변 >
 남자의 수는 577명입니다.


In [60]:
out = agent.invoke({"input": '생존자의 비율은?'})
print("\n답변 >\n", out["output"])


답변 >
 생존자는 342명이고, 전체 인원은 891명입니다. 따라서 생존자의 비율은 약 38.36%입니다. (342 ÷ 891 × 100)


In [59]:
out = agent.invoke({"input": '데이터베이스는 잘 연결되어 있어?'})
print("\n답변 >\n", out["output"])


답변 >
 네, 데이터베이스가 잘 연결되어 있습니다. 현재 경로는 data/titanic.db이며, 연결이 정상적으로 이루어졌습니다.


In [ ]:
print("질문을 입력하세요. (엔터=종료)")
while True:
    q = input("\n질문 > ").strip()
    if not q:
        break
    out = agent.invoke({"input": q})
    print("\n질문 >", q)
    print('-'*100)
    print("\n답변 >\n", out["output"])